In [ ]:
# Empty for "ruff" extension

# Batch algorithm for Near-Duplicates Detection


In [ ]:
import sys
import pyspark.sql.functions as F
from pathlib import Path
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))

from src.spark_utils import get_spark
from src.datasets import read_synthetic

spark = get_spark()

%load_ext autoreload
%autoreload 2

In [ ]:
from src.near_duplicates import partitions_schema
import math


f = 64
p_min = 5
d = 30
probe_stats = []
iterations_needed = []

for K in range(2, int(f // 2)):
    tables = partitions_schema(f=f, K=K, d=d,p_min=p_min)
    # Bits "covered" by the partitions (that are certainly equals)
    partition_bits = sum((table["r"] - K) * table["block_bits"] for table in tables[1])
    combos = tables[0]
    iterations_needed.append(len(tables[1]))
    # Possible matches for each corpus row
    matches = combos * math.pow(2, d / partition_bits)
    probe_stats.append((K, matches))

### PLOT ###
df_stats = pd.DataFrame(
    probe_stats, columns=["K", "Avg matches"]
).astype({"K": str}).set_index("K")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(df_stats.index, df_stats["Avg matches"], marker="o")
ax.set(yscale="log", xlabel="K values", ylabel="Avg matches")
for (x, y), label in zip(df_stats["Avg matches"].items(), iterations_needed): # Number of partitions, over points
    ax.annotate(str(label), (x, y), textcoords="offset points", xytext=(0, 6), ha="center")
ax.grid(True, linestyle="--", alpha=0.5) # Linestyle
ax.margins(y=0.15) # Automargins
plt.xticks(rotation=45)
ax.set_title( # Bottom text
    f"f={f}-bit | corpus=2^{d} | max candidates=2^{p_min}",
    fontsize=10, style="italic", y=-0.25
)
plt.tight_layout()
plt.show()

In [ ]:
from src.fingerprint import create_fingerprints
from src.near_duplicates import detect_near_duplicate_batch, detect_near_duplicate_v2
from src.datasets import read_c5

# Other params are inherited from previous block
f = 64
K = 3
ID_COL = "ID"
INPUT_DATA_COL = "data"

input_df = read_c5()
fingerprints = create_fingerprints(df=input_df, model="bag_of_words", input_col=INPUT_DATA_COL, output_col="fingerprint", fingerprint_size=f, other=None).select(ID_COL, "fingerprint")
print(fingerprints.count())
fingerprints.cache()
# input_df = read_synthetic()
corpus, query = fingerprints.randomSplit([0.8, 0.2])
# result = detect_near_duplicate_batch(corpus, query, fingerprint_size=f, K=K)
result = detect_near_duplicate_v2(corpus, query, f, K, ID_COL, "fingerprint")
'''
result_size = result.count()
result_size
'''
result.show(truncate=False)

In [ ]:
duplicate_count = result.select("corpus_ID").distinct().count()
near_dup_ratio = "{:.2f}".format(duplicate_count / result_size * 100)
print(f"Near-Duplicate elements with maximum hamming distance {K}: {duplicate_count} out of {result_size} ({near_dup_ratio}%)")

In [ ]:
result_sample = result.limit(5)
near_dup_sample = input_df.alias("a").join(F.broadcast(result_sample).alias("b"), F.col(f"a.{ID_COL}") == F.col("b.ID_corpus")).select("b.ID_corpus", "b.ID_query", F.col(f"a.{INPUT_DATA_COL}").alias("corpus_text"))
near_dup_sample = input_df.alias("a").join(F.broadcast(near_dup_sample).alias("b"), F.col(f"a.{ID_COL}") == F.col("b.ID_query")).select("b.ID_corpus", "b.ID_query", "corpus_text", F.col(f"a.{INPUT_DATA_COL}").alias("query_text"))
near_dup_sample.show(truncate=True)

In [ ]:
spark.stop()